# 03 - Tiền xử lý và hợp nhất dữ liệu

Hợp nhất `train`/`test` với `stores`, `oil`, `holidays_events`, `transactions` thành
1 bảng duy nhất theo lưới `(store_nbr, date)`, xử lý các vấn đề dữ liệu thực tế:

- **Giá dầu (`oil.csv`)** chỉ có 1218/1704 ngày (thiếu cuối tuần và 43 ngày rỗng) → tái lập lịch đầy đủ rồi nội suy tuyến tính.
- **Ngày lễ (`holidays_events.csv`)** có 3 phạm vi (`National`/`Regional`/`Local`) và cơ chế *transferred* (lễ bị dời sang ngày khác) → phải quy về đúng 1 cờ `la_ngay_le` cho từng `(store_nbr, date)`.
- **Giao dịch (`transactions.csv`)** chỉ có đến hết giai đoạn train, không có cho giai đoạn test — giữ nguyên (null ở test), không tự chế số liệu.

Toàn bộ logic dưới đây đã được kiểm chứng: số dòng train/test không đổi sau khi ghép
(không bị nhân bản do join), không còn giá trị null ngoài ý muốn.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

THU_MUC_HIEN_TAI = Path.cwd()
GOC = THU_MUC_HIEN_TAI.parent if THU_MUC_HIEN_TAI.name == "notebooks" else THU_MUC_HIEN_TAI
sys.path.append(str(GOC / "src"))

from cau_hinh import DATA_PARQUET, DATA_PROCESSED, tao_thu_muc
from spark_session import tao_spark
from pyspark.sql import functions as F

tao_thu_muc()
spark = tao_spark("03_TienXuLy")

df_train = spark.read.parquet(str(DATA_PARQUET / "train"))
df_test = spark.read.parquet(str(DATA_PARQUET / "test"))
df_stores = spark.read.parquet(str(DATA_PARQUET / "stores"))
df_oil = spark.read.parquet(str(DATA_PARQUET / "oil"))
df_holidays = spark.read.parquet(str(DATA_PARQUET / "holidays_events"))
df_transactions = spark.read.parquet(str(DATA_PARQUET / "transactions"))

so_dong_train_goc = df_train.count()
so_dong_test_goc = df_test.count()
print("So dong train goc:", so_dong_train_goc, "| test goc:", so_dong_test_goc)

## 1. Giá dầu: tái lập lịch đầy đủ + nội suy tuyến tính

Bảng `oil` chỉ có ngày giao dịch (bỏ cuối tuần), lại có 43 dòng giá trị rỗng. Vì
chỉ ~1700 dòng nên xử lý bằng pandas cho gọn (không cần Spark cho dữ liệu nhỏ cỡ này),
sau đó nạp lại vào Spark để join.

In [ ]:
ngay_min = df_train.select(F.min("date")).first()[0]
ngay_max = df_test.select(F.max("date")).first()[0]
print("Khoang ngay can gia dau day du:", ngay_min, "->", ngay_max)

df_oil_pd = df_oil.toPandas()
df_oil_pd["date"] = pd.to_datetime(df_oil_pd["date"])
df_oil_pd = df_oil_pd.set_index("date").reindex(pd.date_range(ngay_min, ngay_max, freq="D"))
df_oil_pd.index.name = "date"
df_oil_pd["gia_dau"] = df_oil_pd["dcoilwtico"].interpolate(method="linear").ffill().bfill()
df_oil_pd = df_oil_pd.reset_index()[["date", "gia_dau"]]
df_oil_pd["date"] = df_oil_pd["date"].dt.date

print(f"Gia dau: {len(df_oil_pd)} ngay day du (goc {df_oil.count()} ngay,",
      f"{df_oil.filter(F.col('dcoilwtico').isNull()).count()} null).",
      f"Con lai {int(df_oil_pd['gia_dau'].isna().sum())} null sau noi suy.")
assert df_oil_pd["gia_dau"].isna().sum() == 0

df_oil_full = spark.createDataFrame(df_oil_pd)

## 2. Ngày lễ / sự kiện đặc biệt theo từng cửa hàng

Quy tắc quy đổi (chuẩn hoá theo mô tả chính thức của bộ dữ liệu Kaggle):

- Loại `Work Day` (5 dòng): ngày bù công cho 1 `Bridge` trước đó → **không** tính là ngày lễ.
- Loại `Holiday` có `transferred = True` (12 dòng): lễ đã bị dời sang ngày khác → ngày gốc
  **không** tính là ngày lễ (ngày dời thực tế nằm ở dòng `Transfer` tương ứng, `transferred = False`).
- Loại `Event` (56 dòng, luôn `National`, ví dụ động đất 2016): là sự kiện đặc biệt có thể
  ảnh hưởng nhu cầu mua sắm, nhưng **không phải ngày nghỉ lễ** → tách thành cờ riêng `la_su_kien_dac_biet`.
- Các dòng còn lại (`Holiday` không transferred, `Additional`, `Bridge`, `Transfer`) được quy
  về cửa hàng theo phạm vi (`locale`): `National` → mọi cửa hàng, `Regional` → cửa hàng có
  `state` khớp `locale_name`, `Local` → cửa hàng có `city` khớp `locale_name`.

In [ ]:
df_hl = df_holidays.filter(~((F.col("type") == "Holiday") & (F.col("transferred") == True)))
df_hl = df_hl.filter(F.col("type") != "Work Day")

df_su_kien = (df_hl.filter(F.col("type") == "Event")
              .select("date").distinct()
              .withColumn("la_su_kien_dac_biet", F.lit(True)))

df_le = df_hl.filter(F.col("type") != "Event")
df_le_quoc_gia = df_le.filter(F.col("locale") == "National").select("date").distinct()
df_le_vung = df_le.filter(F.col("locale") == "Regional").select("date", F.col("locale_name").alias("state"))
df_le_dia_phuong = df_le.filter(F.col("locale") == "Local").select("date", F.col("locale_name").alias("city"))

df_stores_key = df_stores.select("store_nbr", "city", "state")

le_qg = df_stores_key.crossJoin(df_le_quoc_gia).select("store_nbr", "date")
le_vung = df_stores_key.join(df_le_vung, on="state").select("store_nbr", "date")
le_dp = df_stores_key.join(df_le_dia_phuong, on="city").select("store_nbr", "date")

df_ngay_le = le_qg.union(le_vung).union(le_dp).distinct().withColumn("la_ngay_le", F.lit(True))

trung = df_ngay_le.groupBy("store_nbr", "date").count().filter("count > 1").count()
print("So cap (store_nbr, date) bi trung trong bang ngay le:", trung)
assert trung == 0
print("So dong (store_nbr, date) la ngay le:", df_ngay_le.count())
print("So ngay co su kien dac biet:", df_su_kien.count())

## 3. Ghép tất cả thành 1 bảng

In [ ]:
def ghep(df_goc):
    df = df_goc.join(df_stores, on="store_nbr", how="left")
    df = df.join(df_oil_full, on="date", how="left")
    df = df.join(df_ngay_le, on=["store_nbr", "date"], how="left")
    df = df.join(df_su_kien, on="date", how="left")
    df = df.join(df_transactions, on=["store_nbr", "date"], how="left")
    df = df.na.fill({"la_ngay_le": False, "la_su_kien_dac_biet": False})
    return df


df_train_sach = ghep(df_train)
df_test_sach = ghep(df_test)

so_dong_train_sach = df_train_sach.count()
so_dong_test_sach = df_test_sach.count()
print("So dong train sau ghep:", so_dong_train_sach, "(goc:", so_dong_train_goc, ")")
print("So dong test sau ghep:", so_dong_test_sach, "(goc:", so_dong_test_goc, ")")
assert so_dong_train_sach == so_dong_train_goc, "Join lam nhan ban dong o train!"
assert so_dong_test_sach == so_dong_test_goc, "Join lam nhan ban dong o test!"

df_train_sach.printSchema()

## 4. Kiểm tra chất lượng dữ liệu sau ghép

**Lưu ý quan trọng (tránh rò rỉ dữ liệu / data leakage):** cột `transactions` chỉ tồn tại
trong giai đoạn train (Kaggle không cung cấp cho giai đoạn test), nên ở `test_sach` cột
này sẽ **null toàn bộ theo đúng chủ đích** — không được cố gắng điền giá trị hay coi đây là
lỗi dữ liệu. Ở bước đặc trưng (05), nếu dùng `transactions` làm đặc trưng thì chỉ nên dùng
dạng trễ (lag) từ quá khứ, không dùng giá trị ngày hiện tại/tương lai.

In [ ]:
for c in ["gia_dau", "la_ngay_le", "la_su_kien_dac_biet", "city", "state", "cluster"]:
    so_null = df_train_sach.filter(F.col(c).isNull()).count()
    print(f"  null trong {c:<20} (train_sach): {so_null}")
    assert so_null == 0

so_null_transactions_test = df_test_sach.filter(F.col("transactions").isNull()).count()
print("Null transactions trong test_sach (du kien = toan bo):",
      so_null_transactions_test, "/", so_dong_test_sach)
assert so_null_transactions_test == so_dong_test_sach

so_ngay_le = df_train_sach.filter(F.col("la_ngay_le")).count()
print(f"Ty le ngay le trong train: {so_ngay_le}/{so_dong_train_sach} "
      f"({100 * so_ngay_le / so_dong_train_sach:.2f}%)")

## 5. Ghi kết quả vào `data/processed/`

In [ ]:
df_train_sach.write.mode("overwrite").parquet(str(DATA_PROCESSED / "train_sach"))
df_test_sach.write.mode("overwrite").parquet(str(DATA_PROCESSED / "test_sach"))
print("Da ghi:", DATA_PROCESSED / "train_sach")
print("Da ghi:", DATA_PROCESSED / "test_sach")

kt = spark.read.parquet(str(DATA_PROCESSED / "train_sach"))
print("Doc lai train_sach, so dong:", kt.count())
assert kt.count() == so_dong_train_goc

In [ ]:
spark.stop()